
# Lesson 13: BERT Explained - A Complete Guide
*(Based on the article by Samia Khalid)*

## 1. Why was BERT needed?
One of the biggest challenges in NLP is the lack of enough annotated training data. To bridge this gap, researchers developed **pre-training** techniques using enormous amounts of unannotated text from the web.

**BERT (Bidirectional Encoder Representations from Transformers)** is a general-purpose language representation model open-sourced by Google in 2018. Once pre-trained, it can be fine-tuned on smaller, task-specific datasets (like question answering or sentiment analysis) to achieve state-of-the-art accuracy.

## 2. The Core Idea: Bidirectional Context
Before BERT, language models (like Word2Vec) provided **context-free** representations. For example, the word "bank" had the same vector representation in "bank account" and "bank of the river".

Later models added context, but they were largely **unidirectional** (left-to-right) or concatenated two unidirectional models. 

BERT is **deeply bidirectional**. It uses the Transformer architecture to look at both the previous and next context at the exact same time. 
* To predict or understand a word, it immediately pays attention to all other words in the sentence.



## 3. How does it work? Input Embeddings
BERT only uses the **Encoder** part of the Transformer. To process text, BERT requires the input to be decorated with specific metadata:

1.  **Token Embeddings**: A `[CLS]` token is added at the beginning of the first sentence, and a `[SEP]` token is inserted at the end of each sentence.
2.  **Segment Embeddings**: A marker indicating if a token belongs to Sentence A or Sentence B.
3.  **Positional Embeddings**: Indicates the position of each token in the sequence.

The final input embedding is the sum of these three embeddings.
$$ E_{input} = E_{token} + E_{segment} + E_{position} $$

Let's see this in action using the Hugging Face `transformers` library.


In [1]:

# Install required libraries if you haven't already:
# !pip install transformers torch

from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

sentence_a = "I arrived at the bank."
sentence_b = "I crossed the river."

# Encode the sentences
encoded = tokenizer(sentence_a, sentence_b, add_special_tokens=True)

print("Tokens:", tokenizer.convert_ids_to_tokens(encoded['input_ids']))
print("Token IDs:", encoded['input_ids'])
print("Segment IDs (token_type_ids):", encoded['token_type_ids'])

# Notice the [CLS] at the start, and [SEP] separating the two sentences!


Tokens: ['[CLS]', 'I', 'arrived', 'at', 'the', 'bank', '.', '[SEP]', 'I', 'crossed', 'the', 'river', '.', '[SEP]']
Token IDs: [101, 146, 2474, 1120, 1103, 3085, 119, 102, 146, 3809, 1103, 2186, 119, 102]
Segment IDs (token_type_ids): [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]



## 4. Pre-training Strategies

BERT is trained simultaneously on two tasks:

### A. Masked Language Modeling (MLM)
Instead of predicting the next word, BERT randomly masks 15% of the words in the input and tries to predict them based on the context.
Of the 15% chosen for masking:
*   80% are replaced with the `[MASK]` token.
*   10% are replaced with a random token.
*   10% are left unchanged.

Let's use a pre-trained BERT model to perform Masked LM interactively!


In [2]:

from transformers import pipeline

# Load a pre-trained BERT pipeline for Masked Language Modeling
unmasker = pipeline('fill-mask', model='bert-base-uncased')

text = "The woman went to the store and bought a [MASK] of shoes."
predictions = unmasker(text)

print(f"Original: {text}\n")
for idx, pred in enumerate(predictions):
    print(f"Rank {idx+1}: {pred['token_str']} (Score: {pred['score']:.4f})")
    print(f"Result: {pred['sequence']}\n")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original: The woman went to the store and bought a [MASK] of shoes.

Rank 1: pair (Score: 0.9809)
Result: the woman went to the store and bought a pair of shoes.

Rank 2: couple (Score: 0.0080)
Result: the woman went to the store and bought a couple of shoes.

Rank 3: set (Score: 0.0022)
Result: the woman went to the store and bought a set of shoes.

Rank 4: number (Score: 0.0012)
Result: the woman went to the store and bought a number of shoes.

Rank 5: box (Score: 0.0010)
Result: the woman went to the store and bought a box of shoes.




### B. Next Sentence Prediction (NSP)
To understand relationships between sentences (useful for Question Answering), BERT receives pairs of sentences.
*   50% of the time, the second sentence actually follows the first.
*   50% of the time, it is a random sentence.
BERT predicts a binary IsNext-Label using the `[CLS]` token's representation.

## 5. Model Architecture
Google released multiple versions of BERT:
*   **BERT-Base**: 12-layer, 768-hidden-nodes, 12-attention-heads, 110M parameters.
*   **BERT-Large**: 24-layer, 1024-hidden-nodes, 16-attention-heads, 340M parameters.



## 6. Fine-Tuning: Text Classification Tutorial
To fine-tune BERT for a specific task like Sentiment Analysis (e.g., Yelp Reviews), we simply add a single classification layer on top of the pre-trained core model. Specifically, we take the output vector of the `[CLS]` token and pass it through a linear layer to predict our classes.

Below is the modern PyTorch/Hugging Face equivalent of the old `run_classifier.py` script mentioned in the article.


In [3]:

import torch
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

# 1. Load the pre-trained model with a sequence classification head
# For Yelp binary polarity, num_labels = 2
model = BertForSequenceClassification.from_pretrained(
    'bert-base-cased',
    num_labels=2, 
    output_attentions=False,
    output_hidden_states=False,
)

# 2. Prepare Dummy Data (Simulating the Yelp Dataset)
reviews = ["The food was absolutely amazing!", "Terrible service, I will never return."]
labels = [1, 0] # 1 for Good, 0 for Bad

inputs = tokenizer(reviews, padding=True, truncation=True, return_tensors="pt")
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], torch.tensor(labels))
dataloader = DataLoader(dataset, batch_size=2)

# 3. Set up the Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# 4. Minimal Training Loop (One Epoch)
model.train()
for batch in dataloader:
    b_input_ids, b_input_mask, b_labels = batch
    
    # Clear gradients
    model.zero_grad()
    
    # Forward pass
    outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
    loss = outputs.loss
    logits = outputs.logits
    
    print(f"Batch Loss: {loss.item():.4f}")
    
    # Backward pass
    loss.backward()
    optimizer.step()

print("\nTraining step complete! The model weights have been updated based on the classification task.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Batch Loss: 0.7772

Training step complete! The model weights have been updated based on the classification task.
